# 03 — Time-Series Stationarity & Feature Engineering
## Corporación Favorita Grocery Sales Forecasting

### Purpose of this Notebook
This notebook transforms clean tabular data into an advanced, leakage-free time-series feature store:
1. **Stationarity Analysis:** Augmented Dickey-Fuller (ADF) statistical testing, determining optimal differencing order ($d=1$), and applying variance stabilization ($\log(1+x)$).
2. **Autoregressive Lag Features:** Creating $t-1, t-7, t-14, t-28$ lags per `[store_nbr, family]` group.
3. **Rolling & Expanding Window Statistics:** Generating 7-day and 28-day rolling averages and volatility measures using strict `shift(1)` operations to prevent target leakage.
4. **Cyclical Calendar Features:** Implementing trigonometric sine/cosine encodings for weekly and annual seasonality.
5. **Promotional Dynamics:** Shifted promotional lags and 7-day rolling promo intensity.
6. **Holiday Proximity:** Days-to-next-holiday backfill and national holiday flags.
7. **Anomaly Detection:** Unsupervised anomaly labeling using `IsolationForest` to identify demand spikes.
8. **Feature Store Export:** Serializes `feature_store.csv` to `data/` and `data/processed/` for direct consumption by LightGBM and the Streamlit application.


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.ensemble import IsolationForest
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

# --- 1. Universal Path Resolution ---
if os.path.exists('/content/drive/MyDrive/Sales-Forecasting_Project'):
    BASE_DIR = Path('/content/drive/MyDrive/Sales-Forecasting_Project/sales-forecasting')
else:
    cwd = Path.cwd().resolve()
    BASE_DIR = cwd.parent if cwd.name == 'notebooks' else cwd

DATA_DIR = BASE_DIR / 'data'
PROCESSED_DATA_DIR = DATA_DIR / 'processed'
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loading cleaned master data from: {DATA_DIR / 'clean_merged.csv'}...")
df = pd.read_csv(DATA_DIR / 'clean_merged.csv', parse_dates=['date'])
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns.")


### 2. Mathematical Stationarity Testing (Augmented Dickey-Fuller)
Classical models require covariance stationarity (constant mean, constant variance, and autocovariance independent of time).
We test the raw aggregated national sales series.


In [ ]:
daily_sales = df.groupby('date')['sales'].sum().reset_index().set_index('date').asfreq('D')
daily_sales['sales'] = daily_sales['sales'].interpolate()

def test_stationarity(timeseries, alpha=0.05):
    result = adfuller(timeseries.dropna(), autolag='AIC')
    p_value = result[1]
    print(f"ADF Statistic: {result[0]:.4f}")
    print(f"p-value:       {p_value:.4f}")
    print(f"Lags Used:     {result[2]}")
    if p_value <= alpha:
        print(f"Result: p <= {alpha} -> Reject Null Hypothesis. The series is STATIONARY.")
        return True
    else:
        print(f"Result: p > {alpha} -> Fail to reject Null Hypothesis. The series is NON-STATIONARY.")
        return False

print("--- Testing Raw Daily Sales ---")
test_stationarity(daily_sales['sales'])


### 3. Differencing Engine & Variance Stabilization
Because the raw series has a trending mean, we determine the differencing parameter $d$.
We apply first-order differencing ($d=1$) and compare raw differenced sales with log-transformed differenced sales ($\log(1+x)$).


In [ ]:
# First difference
stationary_sales = daily_sales['sales'].diff().dropna()

print("--- Testing First-Differenced Series (d=1) ---")
is_stat = test_stationarity(stationary_sales)

fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
axes[0].plot(daily_sales['sales'], color='#1f77b4')
axes[0].set_title('Raw Sales (Non-Stationary, Trending)', fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(stationary_sales, color='#2ca02c')
axes[1].axhline(0, color='black', linestyle='--', alpha=0.7)
axes[1].set_title('First-Differenced Sales (d=1: Mean Stationary)', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 4. Calendar & Cyclical Feature Engineering
Human behavior follows cycles (7 days in a week, 12 months in a year).
Integer representations (e.g. Month 12 vs Month 1) create an artificial numeric jump.
We project time onto a continuous unit circle using trigonometric transformations:
$$\text{month}_{\sin} = \sin\left(\frac{2\pi \cdot \text{month}}{12}\right), \quad \text{month}_{\cos} = \cos\left(\frac{2\pi \cdot \text{month}}{12}\right)$$
$$\text{dow}_{\sin} = \sin\left(\frac{2\pi \cdot \text{DOW}}{7}\right), \quad \text{dow}_{\cos} = \cos\left(\frac{2\pi \cdot \text{DOW}}{7}\right)$$


In [ ]:
def add_calendar_features(df):
    data = df.copy()
    data['date'] = pd.to_datetime(data['date'])
    data['day_of_week'] = data['date'].dt.dayofweek
    data['month'] = data['date'].dt.month
    data['is_weekend'] = data['day_of_week'].isin([5, 6]).astype(int)
    
    # Payday flags
    data['is_month_start'] = data['date'].dt.is_month_start.astype(int)
    data['is_month_end'] = data['date'].dt.is_month_end.astype(int)
    
    # Cyclical sin/cos encodings
    data['month_sin'] = np.sin(2 * np.pi * data['month'] / 12)
    data['month_cos'] = np.cos(2 * np.pi * data['month'] / 12)
    data['dow_sin'] = np.sin(2 * np.pi * data['day_of_week'] / 7)
    data['dow_cos'] = np.cos(2 * np.pi * data['day_of_week'] / 7)
    
    return data

df = add_calendar_features(df)
print("Calendar & cyclical features created.")
df[['date', 'day_of_week', 'month', 'is_weekend', 'month_sin', 'dow_cos']].head()


### 5. Autoregressive Lag Features (Leakage-Safe)
Autoregressive lags represent past demand. Crucially, rows must be sorted chronologically and grouped by `[store_nbr, family]` so that Store 1's sales are never lagged into Store 2.


In [ ]:
def create_lag_features(df, lags=[1, 7, 14, 28]):
    data = df.sort_values(['store_nbr', 'family', 'date']).copy()
    for lag in lags:
        data[f'sales_lag_{lag}'] = data.groupby(['store_nbr', 'family'])['sales'].shift(lag)
    return data

df = create_lag_features(df, lags=[1, 7, 14, 28])
print("Lag features (1, 7, 14, 28) created.")


### 6. Rolling & Expanding Window Features (Leakage-Safe)
**Golden Rule for Time-Series ML:** Any statistical summary (mean, std dev) for day $t$ must ONLY use information up to day $t-1$.
We strictly apply `.shift(1)` before calling `.rolling()` or `.expanding()`.


In [ ]:
def create_rolling_features(df, windows=[7, 28]):
    data = df.sort_values(['store_nbr', 'family', 'date']).copy()
    
    # Shift FIRST to prevent lookahead leakage
    shifted = data.groupby(['store_nbr', 'family'])['sales'].shift(1)
    
    for w in windows:
        data[f'rolling_mean_{w}'] = shifted.groupby([data['store_nbr'], data['family']]).rolling(window=w, min_periods=1).mean().reset_index(drop=True)
        data[f'rolling_std_{w}'] = shifted.groupby([data['store_nbr'], data['family']]).rolling(window=w, min_periods=1).std().reset_index(drop=True)
        
    # Expanding historical mean
    data['expanding_mean_sales'] = shifted.groupby([data['store_nbr'], data['family']]).expanding().mean().reset_index(drop=True)
    return data

df = create_rolling_features(df, windows=[7, 28])
print("Rolling & expanding window features created.")


### 7. Promotional Features
We engineer 1-day and 7-day lags of promotions, plus 7-day rolling promo intensity.


In [ ]:
def create_promo_features(df):
    data = df.sort_values(['store_nbr', 'family', 'date']).copy()
    promo_shifted = data.groupby(['store_nbr', 'family'])['onpromotion'].shift(1)
    
    data['promo_lag_1'] = promo_shifted
    data['promo_lag_7'] = data.groupby(['store_nbr', 'family'])['onpromotion'].shift(7)
    data['promo_rolling_mean_7'] = promo_shifted.groupby([data['store_nbr'], data['family']]).rolling(window=7, min_periods=1).mean().reset_index(drop=True)
    return data

df = create_promo_features(df)
print("Promotional lag and rolling features created.")


### 8. Holiday Proximity Engineering
We calculate the countdown (`days_to_next_holiday`) by backfilling upcoming holiday dates.


In [ ]:
def add_holiday_proximity(df):
    data = df.sort_values('date').copy()
    
    # Holiday indicators
    data['is_national_holiday'] = ((data['is_holiday'] == 1) & (data['locale'] == 'National')).astype(int)
    
    # Find holiday dates and backfill to compute proximity
    holiday_dates = data['date'].where(data['is_holiday'] == 1)
    next_holiday = holiday_dates.bfill()
    data['days_to_next_holiday'] = (next_holiday - data['date']).dt.days.fillna(999).astype(int)
    return data

df = add_holiday_proximity(df)
print("Holiday proximity features created.")


### 9. Unsupervised Anomaly Detection (Isolation Forest)
We fit an `IsolationForest` on feature attributes to tag statistical outliers (`is_anomaly = 1`) and store the decision score.


In [ ]:
detector_features = ['sales', 'onpromotion', 'dcoilwtico']
clean_subset = df[detector_features].fillna(0)

print("Fitting IsolationForest for anomaly detection...")
iso = IsolationForest(contamination=0.02, random_state=42, n_jobs=-1)
preds = iso.fit_predict(clean_subset)

df['is_anomaly'] = np.where(preds == -1, 1, 0)
df['anomaly_score'] = iso.decision_function(clean_subset)

print(f"Flagged anomalies: {df['is_anomaly'].sum():,} ({df['is_anomaly'].mean()*100:.2f}%)")


### 10. Verification & Feature Store Export
We verify that all required model features are present and non-empty, then export the feature store to both `data/feature_store.csv` and `data/processed/feature_store.csv`.


In [ ]:
REQUIRED_MODEL_FEATURES = [
    'day_of_week', 'month', 'is_weekend', 'is_month_start', 'is_month_end',
    'month_sin', 'month_cos', 'dow_sin', 'dow_cos',
    'sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28',
    'rolling_mean_7', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_28',
    'expanding_mean_sales', 'dcoilwtico',
    'promo_lag_1', 'promo_lag_7', 'promo_rolling_mean_7',
    'is_holiday'
]

missing = [f for f in REQUIRED_MODEL_FEATURES if f not in df.columns]
assert len(missing) == 0, f"Missing features: {missing}"
print(f"All {len(REQUIRED_MODEL_FEATURES)} production model features verified!")

# Save to data/ and data/processed/
target_paths = [
    DATA_DIR / 'feature_store.csv',
    PROCESSED_DATA_DIR / 'feature_store.csv'
]

for p in target_paths:
    print(f"Saving feature store to: {p}...")
    df.to_csv(p, index=False)
    print(f"Saved: {p} ({p.stat().st_size / (1024 * 1024):.2f} MB)")

print("\nFeature Engineering completed successfully! You can now run 07_lightgbm_forecasting.ipynb.")
